# Satisfactory — Power Grid Calculator

Reference data for coal, compacted coal, fuel, and turbo fuel power chains.  
All values from Satisfactory 1.0 (Update 8).

---
**Fuel chain overview:**
```
Coal ──────────────────────────────► Coal Generator (75 MW)
Coal + Sulfur → Compacted Coal ────► Coal Generator (75 MW, fewer items/min)
Crude Oil → Fuel ──────────────────► Fuel Generator (250 MW)
Fuel + Compacted Coal → Turbofuel ─► Fuel Generator (250 MW, 37.5% flow)
```

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from dataclasses import dataclass, field
from typing import Optional

plt.style.use('dark_background')
plt.rcParams.update({
    'figure.figsize': (12, 6),
    'axes.grid': True,
    'grid.alpha': 0.3,
})

## Generator Specs

In [2]:
# ── Coal Generator ──────────────────────────────────────────────────────────
# Accepts: Coal, Compacted Coal, Petroleum Coke
# Always requires water regardless of fuel type
COAL_GEN = {
    'power_mw':                75,
    'coal_per_min':            15.0,
    'compacted_coal_per_min':  50/7,      # ~7.1429 — exact: 4500 MJ/min / 630 MJ each
    'petroleum_coke_per_min':  25.0,
    'water_m3_per_min':        45.0,      # same for all fuel types
}

# ── Fuel Generator ──────────────────────────────────────────────────────────
# Accepts: Fuel, Turbofuel, Liquid Biofuel, Rocket Fuel, Ionized Fuel
FUEL_GEN = {
    'power_mw':                250,
    'fuel_m3_per_min':         20.0,
    'turbofuel_m3_per_min':    7.5,
    'liquid_biofuel_m3_per_min': 20.0,
    'rocket_fuel_m3_per_min':  25/6,     # ~4.1667
    'ionized_fuel_m3_per_min': 3.0,
}

## Energy Values

In [3]:
# MJ per item (solids) or MJ per m³ (liquids)
ENERGY = {
    'coal':            300,    # MJ/item
    'compacted_coal':  630,    # MJ/item  (2.1× coal — 1 coal + 1 sulfur → 1 compacted coal)
    'petroleum_coke':  180,    # MJ/item
    'fuel':            750,    # MJ/m³
    'turbofuel':      2000,    # MJ/m³  (2.67× fuel)
    'liquid_biofuel':  750,    # MJ/m³  (same as fuel)
}

# Quick sanity check — every fuel should hit exactly 4500 MJ/min in a Coal Gen
# and 15000 MJ/min in a Fuel Gen
assert COAL_GEN['coal_per_min']           * ENERGY['coal']           == 4500
assert COAL_GEN['compacted_coal_per_min'] * ENERGY['compacted_coal'] == 4500
assert COAL_GEN['petroleum_coke_per_min'] * ENERGY['petroleum_coke'] == 4500
assert FUEL_GEN['fuel_m3_per_min']        * ENERGY['fuel']           == 15000
assert FUEL_GEN['turbofuel_m3_per_min']   * ENERGY['turbofuel']      == 15000
print('All energy balances pass ✓')

All energy balances pass ✓


## Recipes (per minute)

In [4]:
# Each recipe dict: machine, cycle_time_sec, inputs {item: /min}, outputs {item: /min}

RECIPES = {

    # ── Compacted Coal (Assembler, 12s) ─────────────────────────────────────
    # 1:1 coal:sulfur → 1 compacted coal  (no amplification, but 2.1× energy/item)
    'compacted_coal': {
        'machine':       'Assembler',
        'cycle_sec':     12,
        'inputs':        {'coal': 25, 'sulfur': 25},
        'outputs':       {'compacted_coal': 25},
    },

    # ── Fuel (standard) — Refinery, 6s ──────────────────────────────────────
    'fuel': {
        'machine':       'Refinery',
        'cycle_sec':     6,
        'inputs':        {'crude_oil': 60},
        'outputs':       {'fuel': 40},
        'byproducts':    {'polymer_resin': 30},
    },

    # ── Residual Fuel (alt) — Refinery, 6s ──────────────────────────────────
    # Converts Heavy Oil Residue (byproduct of Plastic/Rubber) to Fuel
    'residual_fuel': {
        'machine':       'Refinery',
        'cycle_sec':     6,
        'inputs':        {'heavy_oil_residue': 60},
        'outputs':       {'fuel': 40},
    },

    # ── Diluted Fuel (alt, hard drive) — Blender, 6s ────────────────────────
    # Best Fuel recipe by far: 100 m³/min from 50 HOR + water
    'diluted_fuel': {
        'machine':       'Blender',
        'cycle_sec':     6,
        'inputs':        {'heavy_oil_residue': 50, 'water': 100},
        'outputs':       {'fuel': 100},
    },

    # ── Turbofuel (standard) — Refinery, 16s ────────────────────────────────
    'turbofuel': {
        'machine':       'Refinery',
        'cycle_sec':     16,
        'inputs':        {'fuel': 22.5, 'compacted_coal': 15},
        'outputs':       {'turbofuel': 18.75},
    },

    # ── Turbo Heavy Fuel (alt) — Refinery, 8s ───────────────────────────────
    # Replaces Fuel input with Heavy Oil Residue directly
    'turbo_heavy_fuel': {
        'machine':       'Refinery',
        'cycle_sec':     8,
        'inputs':        {'heavy_oil_residue': 37.5, 'compacted_coal': 30},
        'outputs':       {'turbofuel': 30},
    },

    # ── Turbo Blend Fuel (alt) — Blender, 8s ────────────────────────────────
    # Best Turbofuel recipe if you have HOR surplus — no compacted coal needed
    'turbo_blend_fuel': {
        'machine':       'Blender',
        'cycle_sec':     8,
        'inputs':        {'fuel': 15, 'heavy_oil_residue': 30, 'sulfur': 22.5},
        'outputs':       {'turbofuel': 45},
        'byproducts':    {'petroleum_coke': 22.5},
    },
}

print('Recipes loaded:', list(RECIPES.keys()))

Recipes loaded: ['compacted_coal', 'fuel', 'residual_fuel', 'diluted_fuel', 'turbofuel', 'turbo_heavy_fuel', 'turbo_blend_fuel']


## Resource Nodes — Sulfur

In [5]:
# Mk.3 Miner output at 100% clock speed (multiply by up to 2.5 with 3 power shards)
SULFUR_NODES = {
    'impure': {'count': 6,  'mk3_per_min': 120},
    'normal': {'count': 5,  'mk3_per_min': 240},
    'pure':   {'count': 5,  'mk3_per_min': 480},
}

max_sulfur_100pct = sum(v['count'] * v['mk3_per_min'] for v in SULFUR_NODES.values())
max_sulfur_250pct = max_sulfur_100pct * 2.5
print(f'Max sulfur @ 100% clock: {max_sulfur_100pct:,.0f} /min')
print(f'Max sulfur @ 250% clock: {max_sulfur_250pct:,.0f} /min')

Max sulfur @ 100% clock: 4,320 /min
Max sulfur @ 250% clock: 10,800 /min


## Clock Speed & Power Shards

In [6]:
# Generators are strictly LINEAR — no exponent like manufacturing buildings
# Each power shard adds 50% capacity (max 3 shards = 250% clock)

POWER_SHARD_CAPS = {0: 1.0, 1: 1.5, 2: 2.0, 3: 2.5}  # multiplier

def gen_output(base_mw: float, clock: float) -> float:
    """Power output at a given clock multiplier (e.g. clock=2.5 for 250%)."""
    return base_mw * clock

def gen_consumption(base_rate: float, clock: float) -> float:
    """Fuel/water consumption at a given clock multiplier."""
    return base_rate * clock

def generators_needed(target_mw: float, base_mw: float, clock: float = 1.0) -> float:
    """How many generators to hit a target MW at the given clock speed."""
    return target_mw / gen_output(base_mw, clock)

# Example: Coal Gens at 250% (3 shards)
coal_gen_250 = gen_output(COAL_GEN['power_mw'], 2.5)
print(f'Coal Gen @ 250%: {coal_gen_250} MW, {gen_consumption(COAL_GEN["coal_per_min"], 2.5):.1f} coal/min, {gen_consumption(COAL_GEN["water_m3_per_min"], 2.5):.1f} m³ water/min')

# Example: Fuel Gens at 250% burning Turbofuel
fuel_gen_250 = gen_output(FUEL_GEN['power_mw'], 2.5)
print(f'Fuel Gen @ 250%: {fuel_gen_250} MW, {gen_consumption(FUEL_GEN["turbofuel_m3_per_min"], 2.5):.4f} m³ turbofuel/min')

Coal Gen @ 250%: 187.5 MW, 37.5 coal/min, 112.5 m³ water/min
Fuel Gen @ 250%: 625.0 MW, 18.7500 m³ turbofuel/min


---
## Your Calculations

Add your factory setup below — generators, resource nodes, chain throughput, etc.

In [7]:
# ── Your setup ──────────────────────────────────────────────────────────────
